# M2 · How an LLM actually works

In M1 you trained a tiny network that picked one of two classes. A large language model is the same idea, just much bigger: at every step it picks one of about 50,000 classes. Each class is a piece of text.

By the end of this lesson you'll have:

- loaded a real language model
- looked inside it
- worked out attention by hand
- written the loop that makes it produce text, the same loop behind every chatbot

**How to work through it** (same as M1)

- Run each cell from top to bottom with **Shift-Enter**, and read the **What each line does** box under it.
- When a cell says **Predict**, write your guess down before running it.
- New words are **bold** where they're explained, and collected in the glossary in `README.md`.
- Save with **Cmd-S** now and then so the dashboard sees your progress.

## 0. What a language model does

A **language model** does one job: given some text, it predicts what comes next. That's all it does, even in ChatGPT.

It doesn't work with letters or whole words. Text is cut into small pieces, and each piece is a **token**: a common word is usually one token, and a rarer word may be split into two or three. The full list of tokens a model knows is its **vocabulary**. Each token in the vocabulary has a number, its *id*.

To write a sentence, the model just repeats one step:

1. Look at all the tokens so far.
2. Score every token in the vocabulary for "how likely is this to come next?"
3. Pick one, stick it on the end, and go back to step 1.

Before we touch a real model, here's that loop with a toy "model". It's just a lookup table that says which word follows which.

In [ ]:
next_word = {
    "the": "cat",
    "cat": "sat",
    "sat": "on",
    "on": "the",
}

text = ["the"]
for _ in range(6):
    text.append(next_word[text[-1]])
print(" ".join(text))

**What each line does**

- `next_word = {...}`: a Python *dictionary*: a lookup table from one word to the word that follows it. This is our toy model.
- `text = ["the"]`: the prompt, the text we start from.
- `for _ in range(6):`: repeat the step 6 times.
- `text[-1]`: the last word so far (`-1` counts from the end).
- `text.append(next_word[...])`: predict the next word and stick it on the end. That's the whole loop.
- A real model replaces the lookup table with a neural network that scores *every* token, using *all* the earlier text, not just the last word.

## 1. Load a real model

We'll use **SmolLM2-135M**, a small open model with 135 million parameters. It's tiny next to ChatGPT, but it's built exactly the same way.

The next cell downloads it from Hugging Face the first time you run it (about 270 MB), then reuses the saved copy.

You need two things:

- The **tokenizer** turns text into token ids, and ids back into text.
- The model itself.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "HuggingFaceTB/SmolLM2-135M"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32)
model.eval()
print("loaded", MODEL)

**What each line does**

- `from transformers import ...`: `transformers` is Hugging Face's library for loading and running models. Nearly every open model can be loaded with it.
- `AutoTokenizer.from_pretrained(MODEL)`: downloads (or reuses) the tokenizer that this model was trained with. A model only understands the tokens of its own tokenizer.
- `AutoModelForCausalLM.from_pretrained(...)`: downloads the model's weights and builds the network. *Causal LM* means a model that predicts the next token.
- `dtype=torch.float32`: keep the weights as 4-byte numbers, so our results match the library's exactly.
- `model.eval()`: puts the model in 'use' mode rather than 'training' mode. Some layers behave slightly differently while training.

**Predict:** how many tokens will `"The capital of France is"` become? Count the words, then run the cell.

In [ ]:
prompt = "The capital of France is"
ids = tokenizer.encode(prompt)
print("token ids:", ids)
print("pieces:   ", tokenizer.convert_ids_to_tokens(ids))
print("back to text:", repr(tokenizer.decode(ids)))
print("vocabulary size:", tokenizer.vocab_size)

**What each line does**

- `tokenizer.encode(prompt)`: text in, list of token ids out. Each id is a row number in the vocabulary.
- `convert_ids_to_tokens(ids)`: shows the piece of text each id stands for. `Ġ` marks a space in front of the word, so ` capital` and `capital` are different tokens.
- `tokenizer.decode(ids)`: ids back to text. `repr(...)` prints it with quotes, so you can see spaces and line breaks.
- `tokenizer.vocab_size`: how many different tokens this model knows: 49,152. Module 3 is all about tokenizers.

In [ ]:
n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params:,} parameters")
print(f"float32: {n_params * 4 / 1e9:.2f} GB    bfloat16: {n_params * 2 / 1e9:.2f} GB")

**What each line does**

- `sum(p.numel() for p in model.parameters())`: counts every adjustable number in the model, as you did for your MLP in M1.
- `{n_params:,}`: prints the number with commas.
- `* 4 / 1e9`: 4 bytes per number, converted to gigabytes (1e9 bytes). The same memory maths as M1 section 1. A 7-billion-parameter model would need about 14 GB in bfloat16.

## 2. Inside the model

SmolLM2 is a **transformer**: the design used by almost every modern language model. More exactly, it's a *decoder-only* transformer, built to predict the next token. A transformer is a stack of identical blocks, and each block does two things:

1. **Attention:** each token gathers information from the tokens before it (section 3).
2. **MLP:** each token's numbers go through a small network, like the one you built in M1.

Before the blocks, each token id is turned into a list of numbers called its **embedding**. After the blocks, a final linear layer turns each token's numbers into one score per vocabulary token. Those scores are the logits.

Print the model to see the pieces:

In [ ]:
print(model)

**What each line does**

- `print(model)`: lists every layer, as `nn.Module` keeps track of them (M1 section 7).
- `embed_tokens: Embedding(49152, 576)`: the embedding table: one row of 576 numbers for each of the 49,152 tokens.
- `layers: (0-29): 30 x LlamaDecoderLayer`: 30 identical blocks, stacked. SmolLM2 has the same design as Meta's Llama models, hence the name.
- Inside each block, `self_attn` is attention, built from `nn.Linear` layers called `q_proj`, `k_proj`, `v_proj` and `o_proj` (section 3 says what q, k and v mean). `k_proj` and `v_proj` are smaller (192 outputs, not 576) because groups of heads share keys and values, a trick that saves memory.
- `mlp` is the small network: three linear layers and `SiLU`, a smoother cousin of ReLU.
- `LlamaRMSNorm` (the `norm` layers): a step that keeps the numbers at a sensible size. `rotary_emb` is how the model knows the *order* of the tokens. You can ignore both for now.
- `lm_head: Linear(in_features=576, out_features=49152)`: the last layer: 576 numbers in, one logit per vocabulary token out. `bias=False` just means these layers have no `b`.

In [ ]:
input_ids = torch.tensor([ids])
with torch.no_grad():
    embeddings = model.model.embed_tokens(input_ids)
print("input_ids:", tuple(input_ids.shape), "  embeddings:", tuple(embeddings.shape))
print("first 5 numbers of the first token's embedding:", embeddings[0, 0, :5])

**What each line does**

- `torch.tensor([ids])`: wraps the list in another list, giving shape (1, 5): one prompt of 5 tokens. Models always take a *batch* of prompts, even if it's just one (M1 section 6).
- `model.model.embed_tokens(input_ids)`: looks up each id's row in the embedding table. It runs inside `torch.no_grad()` because we're only looking, not training; without it PyTorch would print a `grad_fn` note to show it's recording (M1 section 2).
- Shape (1, 5, 576): 1 prompt, 5 tokens, 576 numbers per token. These 576 numbers are what the blocks work on. Training placed similar words close together in this space.

## 3. Attention

The word "it" means different things in "the cat drank because **it** was thirsty" and "the cat drank the milk because **it** was there". To understand a token, the model has to look at other tokens. **Attention** is how it does that.

Think of a library search. Each token makes three lists of numbers from its embedding, each with its own `nn.Linear` layer:

- a **query**: what am I looking for?
- a **key**: what do I contain? (a label other tokens can match against)
- a **value**: the information I'll hand over if someone attends to me

For each token, attention then does four things:

1. **Score** its query against every token's key. The *dot product* (multiply the numbers pairwise, add them up) is big when they match. Divide by √(size) to keep the numbers tame.
2. **Hide the future.** A token may only look at itself and earlier tokens. This is the **causal mask**: put −∞ on every later position, so softmax turns it into 0.
3. **Softmax** each row of scores into weights that add up to 1 (M1 section 5).
4. **Mix:** the output is the weights times the values, a weighted average of the information it attended to.

Here it is with 3 tokens and lists of just 2 numbers, so you can follow every number:

In [ ]:
q = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])  # one query per token
k = torch.tensor([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])  # one key per token
v = torch.tensor([[10.0, 0.0], [0.0, 10.0], [5.0, 5.0]])  # one value per token

scores = q @ k.T / 2**0.5
print("scores:\n", scores)

**What each line does**

- `q`, `k`, `v`: shape (3, 2): 3 tokens, 2 numbers each. In a real model these come from the `q_proj`, `k_proj` and `v_proj` layers; here we just made them up.
- `k.T`: *transposes* k, swapping rows and columns, so (3, 2) becomes (2, 3).
- `q @ k.T`: every query against every key: (3, 2) @ (2, 3) gives (3, 3). Row `i`, column `j` says how well token `i`'s query matches token `j`'s key.
- `/ 2 ** 0.5`: divide by √2, the square root of the list size (2 here, 64 per head in SmolLM2).

In [ ]:
n = 3
mask = torch.triu(torch.ones(n, n, dtype=torch.bool), diagonal=1)
print("mask (True = future, hidden):\n", mask)

scores = scores.masked_fill(mask, float("-inf"))
weights = torch.softmax(scores, dim=-1)
print("weights:\n", weights)
print("output:\n", weights @ v)

**What each line does**

- `torch.ones(n, n, dtype=torch.bool)`: a 3×3 grid of `True`.
- `torch.triu(..., diagonal=1)`: keeps only the part *above* the diagonal (`triu` = upper triangle) and makes the rest `False`. Those are the future positions: row 0 can't see columns 1 and 2.
- `.masked_fill(mask, float("-inf"))`: puts −∞ wherever the mask is `True`. `float("-inf")` is Python's minus infinity.
- `torch.softmax(scores, dim=-1)`: softmax along each row (`dim=-1` means the last dimension). −∞ becomes exactly 0, so the future gets no weight.
- Row 0 of the weights is `[1, 0, 0]`: the first token can only look at itself, so its output is exactly its own value `[10, 0]`.
- `weights @ v`: each output row is a weighted average of the value rows it may see. Token 2 attends a little to everything before it.

That's one **head** of attention. SmolLM2 runs 9 heads side by side in each of its 30 layers, each with its own `q`, `k` and `v`, so different heads can look for different things. Exercise 5 asks you to write one head as a function.

**Predict:** what shape is the score table for a 6-token prompt?

## 4. From logits to the next token

Now run the whole model on the prompt. For every position it outputs **logits**: one score for each of the 49,152 tokens in the vocabulary. They're the same kind of raw class scores as in M1 section 5. The logits at the *last* position are its guess for what comes next.

In [ ]:
with torch.no_grad():
    outputs = model(input_ids)
print("logits shape:", tuple(outputs.logits.shape))

last = outputs.logits[0, -1]
probs = torch.softmax(last, dim=-1)
top = torch.topk(probs, 5)
for p, i in zip(top.values, top.indices, strict=True):
    print(f"{p.item():6.1%}  {tokenizer.decode(int(i))!r}")

**What each line does**

- `with torch.no_grad():`: we're only using the model, not training it, so autograd needn't record anything (M1 section 2). It's faster and uses less memory.
- `model(input_ids)`: runs the whole model. It returns an object with several fields; `.logits` is the one we want.
- Shape (1, 5, 49152): for each of the 5 positions, a score for every vocabulary token. Position `i` predicts token `i + 1`.
- `outputs.logits[0, -1]`: first (only) prompt, last position. Its 49,152 scores are the prediction for the next token. This is exercise 1.
- `torch.softmax(last, dim=-1)`: logits to probabilities (M1 section 5).
- `torch.topk(probs, 5)`: the 5 biggest values and their positions (`.values` and `.indices`). The positions are token ids.
- `tokenizer.decode(int(i))`: turns each token id back into text so you can read it.

The simplest way to choose is **greedy** decoding: always take the most likely token, the argmax of the logits. That's exercise 2.

In [ ]:
next_id = int(last.argmax())
print("greedy choice:", next_id, repr(tokenizer.decode(next_id)))

**What each line does**

- `last.argmax()`: the position of the biggest logit: the most likely next token (M1 section 5).
- `int(...)`: turns the one-number tensor into a plain Python number.

## 5. A generation loop, by hand

Now repeat it: predict, append, predict again. It's the toy loop from section 0, with the real model in place of the lookup table.

Models also have a special **end-of-text token** that means "I'm finished". A generation loop stops early if the model picks it.

In [ ]:
ids_so_far = input_ids
for step in range(8):
    with torch.no_grad():
        logits = model(ids_so_far).logits[0, -1]
    next_id = int(logits.argmax())
    ids_so_far = torch.cat([ids_so_far, torch.tensor([[next_id]])], dim=1)
    print(step, repr(tokenizer.decode(next_id)))
    if next_id == tokenizer.eos_token_id:
        break

print("\n" + tokenizer.decode(ids_so_far[0]))

**What each line does**

- `ids_so_far = input_ids`: start from the prompt's token ids, shape (1, 5).
- `model(ids_so_far).logits[0, -1]`: the next-token logits, as in section 4.
- `int(logits.argmax())`: greedy pick.
- `torch.tensor([[next_id]])`: the new id as a (1, 1) tensor, the same layout as `ids_so_far`.
- `torch.cat([...], dim=1)`: *concatenates*, joining the two tensors along dimension 1 (the token dimension). (1, 5) and (1, 1) give (1, 6), then (1, 7), and so on.
- `tokenizer.eos_token_id`: the id of the end-of-text token. If we pick it, `break` leaves the loop.
- Note the model runs on the *whole* sequence every step. Section 7 shows why that's wasteful.

The `transformers` library has this loop built in as `model.generate()`. In greedy mode it should write exactly the same tokens. Matching it token for token is M2's checkpoint.

In [ ]:
with torch.no_grad():
    library = model.generate(
        input_ids, max_new_tokens=8, do_sample=False, pad_token_id=tokenizer.eos_token_id
    )
print(tokenizer.decode(library[0]))
print("same as ours:", library[0].tolist() == ids_so_far[0].tolist())

**What each line does**

- `model.generate(...)`: the library's generation loop. It returns the prompt plus the new tokens.
- `max_new_tokens=8`: how many tokens to add.
- `do_sample=False`: greedy mode: always the most likely token.
- `pad_token_id=...`: a technical setting for batches of different lengths. It must be set, but it doesn't matter for a single prompt.
- `.tolist()`: turns a tensor into a plain list so `==` compares the whole thing at once and gives one True or False.

## 6. Temperature and top-p

Greedy is predictable: the same prompt always gives the same text, and it often gets stuck repeating itself (you may have just seen it). Chatbots usually use **sampling** instead: pick the next token *at random*, but weighted by its probability, so likely tokens are picked often and unlikely ones rarely.

Two settings control how adventurous sampling is:

- **temperature**: divide the logits by it before softmax. Below 1 makes the top tokens even more likely (sharper, safer). Above 1 spreads the chances out (flatter, more surprising).
- **top-p** (also called *nucleus sampling*): before sampling, throw away the long tail of unlikely tokens. Keep only the most likely tokens whose probabilities add up to at least `p`.

In [ ]:
for t in (0.5, 1.0, 2.0):
    p = torch.softmax(last / t, dim=-1)
    top = torch.topk(p, 3)
    shown = ", ".join(
        f"{tokenizer.decode(int(i))!r} {v.item():.0%}"
        for v, i in zip(top.values, top.indices, strict=True)
    )
    print(f"temperature {t}: {shown}")

**What each line does**

- `last / t`: divide every logit by the temperature. This is exercise 3.
- `torch.softmax(..., dim=-1)`: then turn them into probabilities.
- At 0.5 the top token's share goes up; at 2.0 it goes down. The *order* never changes, only how lopsided it is.
- `", ".join(...)`: glues the three pieces of text together with commas, for printing.

In [ ]:
sorted_probs, order = torch.sort(probs, descending=True)
running_total = torch.cumsum(sorted_probs, dim=-1)
print("top 5 sorted:  ", [round(x, 3) for x in sorted_probs[:5].tolist()])
print("running total: ", [round(x, 3) for x in running_total[:5].tolist()])
print("tokens needed to reach 0.9:", int((running_total < 0.9).sum()) + 1, "of", len(probs))

**What each line does**

- `torch.sort(probs, descending=True)`: sorts biggest first and returns two things: the sorted values and `order`, where each value originally was. You need `order` to map back to token ids.
- `torch.cumsum(...)`: the *cumulative sum*, a running total: 0.262, then 0.262 + 0.094, and so on.
- `(running_total < 0.9).sum() + 1`: how many tokens it takes for the total to reach 0.9. Top-p keeps exactly those and drops the other ~49,000. This is exercise 4.
- `[round(x, 3) for x in ...]`: a *list comprehension*: rounds each number, just for tidy printing.

In [ ]:
gen = torch.Generator().manual_seed(0)
picks = torch.multinomial(probs, num_samples=5, replacement=True, generator=gen)
print("five random draws:", [tokenizer.decode(int(i)) for i in picks])

**What each line does**

- `torch.multinomial(probs, ...)`: draws token ids at random, each with its probability. Likely tokens come up often, unlikely ones rarely.
- `num_samples=5, replacement=True`: five independent draws (the same token may come up twice). Exercise 6 draws one at a time with `num_samples=1`.
- `torch.Generator().manual_seed(0)`: a private random-number source with a fixed seed, so the 'random' draws are the same every run. Exercise 6 passes one in, which is how the checkpoint can test random sampling.

**Predict:** with temperature 2.0, will the five draws be more or less varied? Change `probs` to `torch.softmax(last / 2.0, dim=-1)` in the cell above and see.

## 7. The KV cache

Look back at the loop in section 5. At every step it ran the model on the *whole* sequence again, so by step 20 it reprocesses about 25 tokens just to get one new one.

But the keys and values of earlier tokens never change (the causal mask means they can't see the new tokens). So `generate()` keeps them in memory from step to step: the **KV cache**. Each new step then only has to process the one newest token. The output is identical; it's just much faster.

Here's the difference, timed:

In [ ]:
import time


def tokens_per_second(**settings):
    start = time.perf_counter()
    with torch.no_grad():
        model.generate(
            input_ids,
            max_new_tokens=30,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            **settings,
        )
    return 30 / (time.perf_counter() - start)


print(f"with the KV cache:    {tokens_per_second(use_cache=True):6.1f} tokens/s")
print(f"without it (like ours): {tokens_per_second(use_cache=False):6.1f} tokens/s")

**What each line does**

- `import time` and `time.perf_counter()`: a precise stopwatch. Subtract two readings to get the seconds in between.
- `def tokens_per_second(**settings):`: a small function. `**settings` collects any named settings passed in and hands them on to `generate`.
- `use_cache=True` / `False`: switches the KV cache on or off. Everything else is identical.
- Your hand-written loop works like `use_cache=False`. That's fine for learning; real serving tools always use the cache. For long texts the gap grows, since without it every step reprocesses everything.

## 8. Your task

You've now seen every piece. Time to write them yourself.

1. Open `exercises.py`. There are six exercises, each with plain words and numbered steps, and each points to the section above that shows what you need:
   1. `next_token_logits` (section 4)
   2. `greedy` (section 4)
   3. `softmax_with_temperature` (section 6)
   4. `top_p_filter` (section 6)
   5. `causal_attention` (section 3)
   6. `generate` (sections 5 and 6)
2. Check your work as you go: `make check M=02`. The run tests fail until step 4.
3. Answer the six questions in `answers.toml`.
4. Run your loop on the real model: `uv run python modules/m02_how_llms_work/generate.py`. It runs your `generate()` next to `model.generate()` and checks they write the same 20 tokens.
5. Run `make check M=02` again. When everything passes, M2 turns green on the dashboard.

Stuck? `make hint M=02 EX=4` gives a nudge for exercise 4. Every term is in the glossary in `README.md`.

**Stretch:** try `generate.py --prompt "Once upon a time"`. Then, in this notebook, load the model with `attn_implementation="eager"` and run it with `output_attentions=True` to see which earlier tokens each token attends to.